In [1]:
import pandas as pd
from scipy import stats
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
from sklearn.model_selection import train_test_split
import re
import random
from scipy.stats import zscore
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import make_regression
from sklearn.preprocessing import StandardScaler
import joblib
from scipy.stats import lognorm
from scipy.stats import norm

In [2]:
def log_prob(regrli,scarlerli,x):
    """
    Calculate the log probability of a given set of parameters.

    Parameters:
    x (array): The input parameters. Transformed. 

    Returns:
    float: The log probability of the input parameters.
    """
    # Inverse transform the input parameters
    x = scaler_pars.inverse_transform(x.reshape(1, -1))
    print(x)
    x = pd.DataFrame({
        'p_taper': x[0][0],
        'ball_berry_slope': x[0][1],
        'Vcmax25': x[0][2],
        'sla_top': x[0][3],
        'p50_node_aroot': x[0][4],
        'rs2': x[0][5]
    }, index=[0])
    # Initialize the output
    DFout=pd.DataFrame({})
    # loop through all our model types
    for i in list(range(0,len(Model_List))):
        # load new model/scaler
        regr = regrli[Model_List[i]]
        scaler = scarlerli[Scaler_List[i]]
        obscaler = obli[Obs_List[i]]
        
        ob_set=Obs_save[Obs_save["Set"]==Obs_List[i]]
        if (Obs_List[i] =="LWPmin")|(Obs_List[i] =="LWPmax"):
            my = ob_set[["year", "DOY"]].reset_index(drop=True)
            my["Year"] = my["year"]
            my = my[["Year", "DOY"]]
            ob_set = ob_set.sort_values(["year", "DOY"])
            x_1_run_set = pd.concat([pd.concat([x] * len(my)).reset_index(), my], axis=1)
            one = x_1_run_set.iloc[:, 1:].sort_values(["Year", "DOY"])
            one = one.set_axis(Names2, axis=1)
        else: 
            my = ob_set[["year", "month"]].reset_index(drop=True)
            ob_set = ob_set.sort_values(["year", "month"])
            x_1_run_set = pd.concat([pd.concat([x] * len(my)).reset_index(), my], axis=1)
            one = x_1_run_set.iloc[:, 1:].sort_values(["year", "month"])
            one = one.set_axis(Names, axis=1)
            
        one= scaler.transform(one)
        predicty=regr.predict(one)

        
        if (Obs_List[i] =="LWPmin")|(Obs_List[i] =="LWPmax"):
            predicty=np.abs(predicty)
            
        predicty=obscaler.transform(np.log(predicty).reshape(-1, 1))
        Frame=pd.concat([ob_set.reset_index(drop=True),pd.Series(predicty.flatten(),name="sim").reset_index(drop=True)],axis=1)
        if (Obs_List[i] =="LWPmin")|(Obs_List[i] =="LWPmax"):
            Frame=Frame.sample(n=169,random_state=55)                    
        #print(Frame)
     #### This is a dataframe with the obs/sim/ in it
        DFout=pd.concat([DFout,Frame])
    #print(DFout)
    addsim=DFout.loc[DFout["Set"]=='MonthlyGPP']["sim"]/DFout.loc[DFout["Set"]=='ET']["sim"]
    addobs=DFout.loc[DFout["Set"]=='MonthlyGPP']["obs"]/DFout.loc[DFout["Set"]=='ET']["obs"]
    #### Calculate full ll
    ll1=np.sum(norm.logpdf(DFout['sim'], loc=DFout["obs"], scale=5.0))
    ll2=np.sum(norm.logpdf(addsim, loc=addobs, scale=5.0))
    ll1+=ll2
    # Check physical possibility/Values are within the bounds of initial sample. 
    p =sum([1 for t in range(len(x.columns)) if (x.iloc[0, t] < samples[[x.columns[t]]].min().values) or (x.iloc[0, t] > samples[[x.columns[t]]].max().values)])
    if p >=1:
        ll1= -np.inf
    return (ll1)

def boundby(single,min,max):
    if single<min :
        single=min+np.abs(min-single)
    if single>max:
        single=max-(single-max)
    if single<min :
        single=min+np.abs(min-single)
    if single>max:
        single=max-(single-max)
    if single>max:
       single=np.random.uniform(min,max)
    if single<min:
       single=np.random.uniform(min,max)
    return(single)

def bounce(newproposed,samples_scale):
    for i in list(range(0,len(samples_scale.columns))):### Maybe this isn't minus 1
        newproposed[i]=boundby(newproposed[i],samples_scale.iloc[:,i].min(),samples_scale.iloc[:,i].max())
    return(newproposed)
def acceptance(x_likelihood, x_new_likelihood):
  if x_new_likelihood>x_likelihood:
        print("True_accept")
        return True
  else:
        accept=np.random.uniform(0,1)
        # Since we did a log likelihood, we need to exponentiate in order to compare to the random number
        # less likely x_new are less likely to be accepted
        print(np.exp(x_new_likelihood-x_likelihood))
        if accept < np.exp(x_new_likelihood-x_likelihood):
            print("roll accept")
            return True
        else:
            return False
        #return(np.exp(x_likelihood-x_new_likelihood))

def Loadmodels(LOADIN,EmDir,Model_List,Scaler_List):
    if LOADIN== True: 
        regrli={}
        scarlerli={}
        for i in list(range(0,len(Model_List))) :
            print(i)
         # load new model/scaler
            variable_name=Model_List[i]
            regrli[Model_List[i]]=joblib.load(Emdir+Model_List[i]+'.joblib')
            print(regrli)
            scarlerli[Scaler_List[i]]=joblib.load(Emdir+Scaler_List[i]+'.joblib')
        return(regrli,scarlerli) 
#regrli,scarleri=Loadmodels(LOADIN=True,
#"C:/Users/345578/Desktop/NewML/ML_models/",
 #       ["ET_full_012825", "GPP_minvar_012825",'H2oSOI1_minvar_012825',"LWP_max_model_012825",
 #            "LWP_min_model_012825","RO_minvar_031125"],["ET_full_012825_Scalar","GPP_minvar_031125_Scalar","H2oSOI1_minvar_012825_Scalar",
 #             "LWP_max_model_012825_Scalar","LWP_min_model_012825_Scalar","RO_minvar_031125_Scalar"]

def CleanScaleObs(Obs_save):
    Obs_save=Obs_save.dropna()
    Obs_save['Date']=pd.to_datetime(Obs_save["Date"])
    Obs_save["year"]=Obs_save["Date"].dt.year
    Obs_save["month"]=Obs_save["Date"].dt.month
    Obs_save["DOY"]=Obs_save["Date"].dt.dayofyear
    #### Create scaler and scale and scale
    GPP_obsscale=StandardScaler().fit(np.log(np.abs(np.array(Obs_save.loc[Obs_save["Set"]==list1[0],'obs']))).reshape(-1, 1))
    RO_obsscale=StandardScaler().fit(np.log(np.abs(np.array(Obs_save.loc[Obs_save["Set"]==list1[1],'obs']))).reshape(-1, 1))
    SWC_obsscale=StandardScaler().fit(np.log(np.abs(np.array(Obs_save.loc[Obs_save["Set"]==list1[2],'obs']))).reshape(-1, 1))
    ET_obsscale=StandardScaler().fit(np.log(np.abs(np.array(Obs_save.loc[Obs_save["Set"]==list1[5],'obs']))).reshape(-1, 1))
    Min_obsscale=StandardScaler().fit(np.log(np.abs(np.array(Obs_save.loc[Obs_save["Set"]==list1[6],'obs']+0.0001))).reshape(-1, 1))
    Max_obsscale=StandardScaler().fit(np.log(np.abs(np.array(Obs_save.loc[Obs_save["Set"]==list1[7],'obs']+0.0001))).reshape(-1, 1))
    Obs_save.loc[Obs_save["Set"]==list1[0],'obs']=GPP_obsscale.transform(np.log(np.array(Obs_save.loc[Obs_save["Set"]==list1[0],'obs']).reshape(-1, 1)))
    Obs_save.loc[Obs_save["Set"]==list1[1],'obs']=RO_obsscale.transform(np.log(np.array(Obs_save.loc[Obs_save["Set"]==list1[1],'obs']).reshape(-1, 1)))
    Obs_save.loc[Obs_save["Set"]==list1[2],'obs']=SWC_obsscale.transform(np.log(np.array(Obs_save.loc[Obs_save["Set"]==list1[2],'obs']).reshape(-1, 1)))
    Obs_save.loc[Obs_save["Set"]==list1[5],'obs']=ET_obsscale.transform(np.log(np.array(Obs_save.loc[Obs_save["Set"]==list1[5],'obs']).reshape(-1, 1)))
    Obs_save.loc[Obs_save["Set"]==list1[6],'obs']=Min_obsscale.transform(np.log(np.abs(np.array(Obs_save.loc[Obs_save["Set"]==list1[6],'obs']+0.0001))).reshape(-1, 1))
    Obs_save.loc[Obs_save["Set"]==list1[7],'obs']=Max_obsscale.transform(np.log(np.abs(np.array(Obs_save.loc[Obs_save["Set"]==list1[7],'obs']+0.0001))).reshape(-1, 1))
    
    Obi = ["ET",'MonthlyGPP', 'SWC10',"LWPmax","LWPmin",'Runoff']
    obli={}
    obli[Obi[0]]=ET_obsscale
    obli[Obi[1]]=GPP_obsscale
    obli[Obi[2]]=SWC_obsscale
    obli[Obi[3]]=Max_obsscale
    obli[Obi[4]]=Min_obsscale
    obli[Obi[5]]=RO_obsscale
    
    return(Obs_save, obli)

def TestLL(regrli,scarleri,initialpos=225):
    #initial_positin=#[samples_sub.iloc[[initialpos]].T,samples_sub]
    initial_position=np.array(samples_sub.iloc[[initialpos]].T)
    print(initial_position)
    Logl=log_prob(regrli,scarleri,scaler_pars.transform(np.array(initial_position).flatten().reshape(1, -1)).flatten())
    print(Logl)


def AdaptiveMCMC(par_cov_matrix,steps,adapt_interval,burnin,dim,x_1,benchmark="./save.csv"):
    '''
    par_cov_matrix: initial step matrix
    steps: number of steps to take in total
    adapt_interval: How often to adapt the sampling range
    burnin: How long before counting adjustments
    dim: Dimensions in the parameter set
    x_1: inital parameters in this script ex samples_sub.iloc[[250]].T
    benchmark: where to save the benchmarks for saving. 

    returns movelog, acceptlog, scorelog
    '''
    
    movelog=pd.DataFrame([])
    acceptlog=pd.DataFrame([])
    scorelog=np.array([])
    namelog=np.array([])
    acceptance_rates = []
    num_adapted_samples=0
   
    x_1=scaler_pars.transform(np.array(x_1).flatten().reshape(1, -1)).flatten()
    Logl=log_prob(regrli,scarleri,x_1)
    print("Initial Logliklihood")
    print(Logl)
    sum_states = np.zeros(dim)
    outer_product_sum = np.zeros((dim, dim))
    num_adapted_samples = 0
    lag=0
    plus=0
    accepted_samples_for_adaptation=[]
    Trueburn=burnin+adapt_interval
    for time in list(range(1,steps)):
        print(time)
        lag=lag+1
        if time >= burnin and (time + 1) % adapt_interval == 0 and (len(accepted_samples_for_adaptation) > 1):
                # Update running sums for mean and covariance calculation
                   #print(accepted_samples_for_adaptation)
                    par_cov_matrix= (np.cov(np.array(accepted_samples_for_adaptation).T))*.5
                    print(par_cov_matrix)
                    # Add a small regularization to prevent singular covariance matrix
                    par_cov_matrix += np.eye(dim) * 1e-6
                   
    
        newproposed=np.random.multivariate_normal(np.array(x_1),par_cov_matrix, size=1).tolist()[0]
        newproposed=bounce(newproposed,pd.DataFrame(samples_scale))
        movelog=pd.concat([movelog,pd.DataFrame(newproposed)],axis=1)
        NewLogl=log_prob(regrli,scarleri,np.array(newproposed))
        NewLogl=NewLogl+plus
        print(Logl)
        print(NewLogl)  
        scorelog=np.append(scorelog,Logl)
        if (acceptance(Logl,NewLogl)):
            Logl=NewLogl-plus
            plus=0
            lag=0
            
            x_1=newproposed
            accepted_samples_for_adaptation.append(newproposed)
            print('Accept')
            print(len(accepted_samples_for_adaptation))
            if time >Trueburn:
                acceptlog=pd.concat([acceptlog,pd.DataFrame(newproposed)],axis=1)
        if lag==20:
            lag=0
            plus=plus+100
       #     x_1=newproposed
       #     accepted_samples_for_adaptation.append(newproposed)
       #     print('Accept')
       #     print(len(accepted_samples_for_adaptation))
       #     if time >Trueburn:
       #         acceptlog=pd.concat([acceptlog,pd.DataFrame(newproposed)],axis=1)
        if time % 500 == 0:
            pd.DataFrame(movelog).to_csv(benchmark)
    print("Total Accepted")
    print(len(acceptlog.T))
    return(movelog)


In [3]:
Emdir = "C:/Users/345578/Desktop/NewML/ML_models/"
Obsdir="C:/Users/383517/Desktop/LT2024/Machine_Learning/FATES_emulator/"
Datadir="C:/Users/383517/Desktop/LT2024/Machine_Learning/Data/"
Model_List = ["ET_full_012825", "GPP_minvar_012825",'H2oSOI1_minvar_012825',"LWP_max_model_012825",
             "LWP_min_model_012825","RO_minvar_031125"]
Scaler_List = ["ET_full_012825_Scalar","GPP_minvar_031125_Scalar","H2oSOI1_minvar_012825_Scalar",
              "LWP_max_model_012825_Scalar","LWP_min_model_012825_Scalar","RO_minvar_031125_Scalar"]
Obs_List = ["ET",'MonthlyGPP', 'SWC10',"LWPmax","LWPmin",'Runoff']
list1=['MonthlyGPP', 'Runoff', 'SWC10', 'SWC40', 'SWC100', 'ET', 'LWPmin',
       'LWPmax']



In [4]:
samples=pd.read_csv('C:/Users/345578/Desktop/ML_scaler/LHS.sam.csv')
Varset= ['p_taper',
       'ball_berry_slope', 'Vcmax25', 'sla_top', 'p50_node_aroot','rs2']
samples_sub=samples[Varset]
scaler_pars =StandardScaler().fit(samples_sub.values)
samples_scale=scaler_pars.transform(samples_sub)
Names=pd.concat([pd.Series(samples_sub.columns),pd.Series(["year","month",])])
Names2=pd.concat([pd.Series(samples_sub.columns),pd.Series(["Year","DOY",])])

Obs_save=pd.read_csv("C:/Users/345578/Desktop/NewML/Synth/Syntheticg_Set1119.csv").iloc[:,1:]
Obs_save=Obs_save.dropna()
Obs_save,obli=CleanScaleObs(Obs_save)

C:\Users\345578\Anaconda3\envs\lt\Lib\site-packages\sklearn\base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [5]:
accepted_samples_for_adaptation

NameError: name 'accepted_samples_for_adaptation' is not defined

In [ ]:
LOADIN=True
if LOADIN==True:
    regrli,scarleri=Loadmodels(LOADIN,Emdir,Model_List,Scaler_List)

TestLL(regrli,scarleri,initialpos=1118)



0
{'ET_full_012825': RandomForestRegressor()}
1
{'ET_full_012825': RandomForestRegressor(), 'GPP_minvar_012825': RandomForestRegressor()}
2
{'ET_full_012825': RandomForestRegressor(), 'GPP_minvar_012825': RandomForestRegressor(), 'H2oSOI1_minvar_012825': RandomForestRegressor()}
3
{'ET_full_012825': RandomForestRegressor(), 'GPP_minvar_012825': RandomForestRegressor(), 'H2oSOI1_minvar_012825': RandomForestRegressor(), 'LWP_max_model_012825': RandomForestRegressor()}
4


In [ ]:
initialpos=45
S=[1,1,1,1,1,1]
steps=52000
adapt_interval=1000
burnin=1000
dim=6



par_cov_matrix=np.outer(S,S)
print(par_cov_matrix)
log=AdaptiveMCMC(par_cov_matrix,
             steps,
             adapt_interval,
             burnin,dim,
             samples_sub.iloc[[initialpos]].T,
             benchmark="C:/Users/345578/Desktop/NewML/MCMC_runs/bm.csv")

[[1 1 1 1 1 1]
 [1 1 1 1 1 1]
 [1 1 1 1 1 1]
 [1 1 1 1 1 1]
 [1 1 1 1 1 1]
 [1 1 1 1 1 1]]
[[2.67227319e-01 1.34776875e+01 1.05070461e+02 3.19442435e-02
  1.86562101e+00 1.15573906e-01]]
Initial Logliklihood
-10959.034625908695
1
[[4.13600161e-01 1.43396946e+01 1.03493288e+02 4.50670508e-02
  3.90893350e+00 3.38143493e-01]]
-10959.034625908695
-9454.622137088954
True_accept
Accept
1
2
[[3.98801593e-01 1.39168583e+01 1.00322159e+02 4.35519646e-02
  3.70235079e+00 3.15641291e-01]]
-9454.622137088954
-395121.17477915867
0.0
3
[[3.75290985e-01 1.32450949e+01 9.52841590e+01 4.11449344e-02
  3.37415109e+00 2.79891855e-01]]
-9454.622137088954
-34217.230498733275
0.0
4
[[4.04292950e-01 1.24624790e+01 9.74567395e+01 2.83065979e-02
  6.44824068e+00 6.14739732e-01]]
-9454.622137088954
-65083.3997014887
0.0
5
[[4.93070452e-01 1.49990989e+01 1.16480535e+02 3.73956910e-02
  5.20893856e+00 4.79747673e-01]]
-9454.622137088954
-9253.657585192628
True_accept
Accept
2
6
[[2.69141396e-01 8.60082318e+00 6.

In [45]:
Fixedlog=pd.DataFrame(scaler_pars.inverse_transform(log.T))
Fixedlog.to_csv("C:/Users/345578/Desktop/NewML/MCMC_runs/TestMC86_2.csv")

In [ ]:
for i in list(range(0,6)):
    times=2
   # print(V["b"][i])
   # print(V["SD"][i]*times)
    plt.acorr(Fixedlog.iloc[1000:1100,i],maxlags=98) 
    plt.show()
    plt.plot(Fixedlog.iloc[:,i])
    plt.show()